# 06 — Crosswalk and master table

**Owner:** F  
**Goal:** one row per municipality per election, with the funnel columns

**Reads:** `data/interim/*.csv`  
**Writes:** `PROCESSED / "master.csv"`

Run the two setup cells below first, every time.

## Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:                                    # private repo: token from Colab Secrets
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        print("No GITHUB_TOKEN in Colab Secrets - cloning will fail for the private repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

## Crosswalk

Codes present in one year but not another — map them by hand, with a note per row.

In [ ]:
# cw = pd.read_csv(INTERIM / "muni_crosswalk.csv")
# set(df11.muni_code) - set(df16.muni_code)

## Merge everything on muni_code

Results + registration + signals + census. Left join on results.

In [ ]:
# master = (results.merge(reg, on="muni_code", how="left")
#                  .merge(signals, on="muni_code", how="left")
#                  .merge(census, on="muni_code", how="left"))

## Drop districts, keep district_code

Districts would double-count voters.

In [ ]:
# master = master[master["muni_type"] != "DC"]

## The funnel

r = R/E, t = V/R, p = r*t, then the two leaks against the national benchmark.

In [ ]:
import numpy as np
# master["r"] = master["registered"] / master["eligible_adults"]
# master["t"] = master["votes_cast"] / master["registered"]
# master["p"] = master["r"] * master["t"]
# r_star, t_star = master["r"].median(), master["t"].median()
# master["leak_reg"]  = np.log(master["r"] / r_star)
# master["leak_turn"] = np.log(master["t"] / t_star)

## Spatial lag

Mean 2021 turnout of neighbouring municipalities.

In [ ]:
# master["spatial_lag_turnout"] = ...

## Checks

Missing values per column, and any r above 1.05 (Census undercount — flag, do not drop).

In [ ]:
# print(master.isna().sum())
# master.loc[master["r"] > 1.05, ["muni_name","r"]]

## Save output

In [ ]:
# out.to_csv(PROCESSED / "master.csv", index=False)
# print(out.shape); out.head()